# 04 Topography

**Project:** Pine Ridge Bison Habitat Suitability Analysis  
**BHSI Component:** Topography (15% weight)  
**Data Source:** USGS 3DEP 1/3 arc-second DEM

## Why Topography Matters for Bison

Bison are plains animals. They evolved on the flat and gently rolling
grasslands of the Northern Great Plains and strongly prefer terrain
with slope < 15 degrees for grazing and daily movement.

Steep terrain has three management implications:
- Bison avoid it for sustained grazing as it reduces effective habitat area
- It concentrates animal pressure on adjacent flat areas
- It increases erosion risk under grazing pressure

On Pine Ridge, the primary topographic constraint is the White River
Badlands along the northern and western edges of the reservation so deeply
dissected terrain with slopes far exceeding bison preferences.

This notebook computes slope from the USGS 3DEP DEM and converts it
to a suitability score. Aspect (north vs. south facing) is also computed
as a secondary modifier for snow retention and forage productivity.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling

from src.constants import (
    CRS_PROJECTED, PINE_RIDGE_BBOX,
    SLOPE_THRESHOLDS, CACHE_DIR, OUTPUTS_DIR, FIGURES_DIR,
)
from src.loaders import load_srtm_dem
from src.raster_utils import normalize_0_1, align_raster_to_template
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore")
%matplotlib inline

TEMPLATE_PATH = CACHE_DIR/"template_30m_albers.tif"
assert TEMPLATE_PATH.exists(), "Run notebook 01 first."
pine_ridge = gpd.read_file(OUTPUTS_DIR/"pine_ridge_boundary.geojson")
print("Setup complete.")
print(f"Slope thresholds: {SLOPE_THRESHOLDS}")

In [ ]:
# Print the data sovereignty statement at the top of every notebook
print_data_acknowledgment(source_keys=["srtm"])

## Download and Process DEM

In [ ]:
dem_path = load_srtm_dem(bbox=PINE_RIDGE_BBOX)

if dem_path is None:
    print("SRTM DEM not available. Creating synthetic slope surface for demo.")
    # Fallback: uniform moderate suitability
    with rasterio.open(TEMPLATE_PATH) as tmpl:
        tmpl_data = tmpl.read(1)
        profile   = tmpl.profile.copy()
    slope_arr = np.where(~np.isnan(tmpl_data), 5.0, np.nan).astype(np.float32)
    DEM_AVAILABLE = False
else:
    print(f"DEM file: {dem_path}")
    with rasterio.open(dem_path) as src:
        print(f"DEM shape : {src.read(1).shape}")
        print(f"DEM CRS   : {src.crs}")
        print(f"DEM res   : {src.res}")
    DEM_AVAILABLE = True

In [ ]:
if DEM_AVAILABLE:
    # Reproject DEM to Albers Equal Area and align to template
    dem_aligned = CACHE_DIR/"dem_aligned.tif"
    align_raster_to_template(
        src_path=dem_path,
        template_path=TEMPLATE_PATH,
        output_path=dem_aligned,
        resampling_method="bilinear",
    )

    # Compute slope using numpy gradient
    with rasterio.open(dem_aligned) as src:
        elev   = src.read(1).astype(np.float32)
        if src.nodata is not None:
            elev[elev == src.nodata] = np.nan
        res_m  = abs(src.transform.a)

    # Gradient in x and y directions -> slope in degrees
    dy, dx    = np.gradient(elev, res_m, res_m)
    slope_arr = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

    valid_mask = (~np.isnan(elev)) & (~np.isnan(tmpl_data))
    coverage = valid_mask.sum() / (~np.isnan(tmpl_data)).sum()
    if coverage < 0.99:
        raise ValueError(f"DEM covers only {coverage:.1%} of the analysis area; refresh the SRTM download before scoring.")
    slope_arr[~valid_mask] = np.nan
    valid = slope_arr[valid_mask]
    print(f"Slope statistics (degrees):")
    print(f"  Min   : {np.nanmin(slope_arr):.1f}°")
    print(f"  Max   : {np.nanmax(slope_arr):.1f}°")
    print(f"  Mean  : {np.nanmean(slope_arr):.1f}°")
    print(f"  Median: {np.nanmedian(slope_arr):.1f}°")
    print()
    for label, thresh in [
        ("Ideal (< 5°)",       SLOPE_THRESHOLDS["ideal"]),
        ("Good (5-15°)",       SLOPE_THRESHOLDS["good"]),
        ("Marginal (15-25°)",  SLOPE_THRESHOLDS["marginal"]),
    ]:
        pct = (slope_arr[~np.isnan(slope_arr)] < thresh).mean() * 100
        print(f"  {label}: {pct:.1f}% of pixels")

## Convert Slope to Suitability Score

In [ ]:
# Score function: linear decline from ideal to marginal
# Beyond marginal (>25 degrees) = 0
with rasterio.open(TEMPLATE_PATH) as tmpl:
    tmpl_data = tmpl.read(1)
    profile   = tmpl.profile.copy()

ideal    = SLOPE_THRESHOLDS["ideal"]
good     = SLOPE_THRESHOLDS["good"]
marginal = SLOPE_THRESHOLDS["marginal"]
exclude  = SLOPE_THRESHOLDS["exclude"]

slope_suit = np.where(
    slope_arr <= ideal,   1.00,
    np.where(
        slope_arr <= good,
        1.00 - 0.40 * (slope_arr - ideal) / (good - ideal),
        np.where(
            slope_arr <= marginal,
            0.60 - 0.55 * (slope_arr - good) / (marginal - good),
            0.05
        )
    )
).astype(np.float32)

# Apply boundary mask
slope_suit[np.isnan(tmpl_data)] = np.nan

topo_path = OUTPUTS_DIR / "bhsi_topography.tif"
with rasterio.open(topo_path, "w", **profile) as dst:
    dst.write(slope_suit, 1)

valid = slope_suit[~np.isnan(slope_suit)]
print(f"Topography suitability layer:")
print(f"  Mean  : {valid.mean():.3f}")
print(f"  Pixels with score >= 0.8: {(valid >= 0.8).mean()*100:.1f}%")
print(f"  Saved : outputs/bhsi_topography.tif")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if DEM_AVAILABLE:
    im0 = axes[0].imshow(slope_arr, cmap="terrain_r",
                          vmin=0, vmax=30, origin="upper")
    plt.colorbar(im0, ax=axes[0], label="Slope (degrees)", shrink=0.8)
    axes[0].set_title("Slope from SRTM GL3 DEM", fontsize=10, fontweight="bold")
else:
    axes[0].text(0.5, 0.5, "DEM not available\n(fallback used)",
                 ha="center", va="center", transform=axes[0].transAxes)
    axes[0].set_title("Slope (unavailable)", fontsize=10)

im1 = axes[1].imshow(slope_suit, cmap="RdYlGn",
                      vmin=0, vmax=1, origin="upper")
plt.colorbar(im1, ax=axes[1],
             label="Topographic suitability (0-1)", shrink=0.8)
axes[1].set_title("Topographic Suitability", fontsize=10, fontweight="bold")

for ax in axes:
    ax.set_xlabel("Column (west to east)")
    ax.set_ylabel("Row (north to south)")

plt.suptitle(
    "BHSI Component 3: Topography (15% weight)\n"
    "Pine Ridge Reservation Slope-Based Suitability",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
fig.savefig(FIGURES_DIR/"04_topography.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(generate_citations(["srtm"]))